# 01a — Load and clean 3-D hologram stacks

Load one detector acquisition without collapsing its frame axis, subtract an averaged and optionally linearly fitted dark, inspect its intensities, threshold the corrected frames, average them, and save one compact 2-D image using its acquisition ID.

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())
    
BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)

from data_loading import SextantsNexusLoader, load_processing
print("Base folder:", BASEFOLDER)


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

%matplotlib qt

/usr/lib/python3/dist-packages/pytools/persistent_dict.py:52: RecommendedHashNotFoundWarning: Unable to import recommended hash 'siphash24.siphash13', falling back to 'hashlib.sha256'. Run 'python3 -m pip install siphash24' to install the recommended hash.
  warn("Unable to import recommended hash 'siphash24.siphash13', "


Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW


## Configuration

In [2]:

RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"

IMAGE_ID = 550
DARK_IDS = [548,549]


## Load the 3-D arrays

In [3]:
loader = SextantsNexusLoader(RAW_FOLDER)

# load_processing returns the simple 2-D average and the full 3-D stack.
blind_average, stack = load_processing(loader, IMAGE_ID)

# The dark is also a 3-D acquisition. Use its simple frame average.
dark_reference, dark_stack = load_processing(loader, DARK_IDS)
dark_reference = np.asarray(dark_reference, dtype=float)
print(f"Loaded ID {IMAGE_ID}: stack {stack.shape}, average {blind_average.shape}")
print("Dark stack:", dark_stack.shape, "dark average:", dark_reference.shape)

Loaded ID 550: stack (101, 2048, 2048), average (2048, 2048)
Dark stack: (202, 2048, 2048) dark average: (2048, 2048)


## Dark subtraction

Fit `frame ≈ scale × dark + offset` on low-intensity pixels from a small selected detector region. The fitted dark background is then subtracted from every pixel of the complete frame.

In [4]:
plt.close("all")


# Dark-rescaling controls are here because they apply to this operation.
FIT_DARK_LINEAR = True
DARK_FIT_PERCENTILE = 100
DARK_FIT_ROWS = slice(0, 100)
DARK_FIT_COLUMNS = slice(0, 100)
DARK_FIT_STRIDE = 1  # Optional subsampling within the selected region.



def subtract_fitted_dark(frame, dark, rows, columns, percentile=30, stride=1):
    # Estimate scale and offset only from the selected detector region.
    sample_image = frame[rows, columns][::stride, ::stride].ravel().astype(float)
    sample_dark = dark[rows, columns][::stride, ::stride].ravel().astype(float)
    valid = np.isfinite(sample_image) & np.isfinite(sample_dark)
    limit = np.percentile(sample_image[valid], percentile)
    valid &= sample_image <= limit
    if valid.sum() < 2 or np.ptp(sample_dark[valid]) == 0:
        scale, offset = 1.0, 0.0
    else:
        scale, offset = np.polyfit(sample_dark[valid], sample_image[valid], 1)
    corrected = frame - (scale * dark + offset)
    return corrected.astype(np.float32), float(scale), float(offset)

corrected_frames = []
dark_fits = []
for frame in stack:
    if FIT_DARK_LINEAR:
        corrected, scale, offset = subtract_fitted_dark(
            frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
            DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
        )
    else:
        corrected, scale, offset = frame - dark_reference, 1.0, 0.0
    corrected_frames.append(corrected)
    dark_fits.append((scale, offset))
corrected_stack = np.stack(corrected_frames)
dark_fits = np.asarray(dark_fits)
print("Dark scale mean:", dark_fits[:, 0].mean(),
      "offset mean:", dark_fits[:, 1].mean())

Dark scale mean: 0.9991776132755715 offset mean: 5.19185582716225


In [14]:
cimshow(corrected_stack)

interactive(children=(FloatRangeSlider(value=(-108.23196554565429, 36749.1470429725), description='contrast', …

interactive(children=(IntSlider(value=0, description='nr'), Output()), _dom_classes=('widget-interact',))

(<Figure size 700x700 with 1 Axes>, <Axes: >)

## Inspect the dark rescaling fit

The scatter plot uses the first frame of the selected image. Grey points are all finite sampled pixels, blue points are the pixels used for the linear fit, and the red line is the fitted dark background.

In [12]:
# Show the same pixels and selection used for the first frame's fit.
image_values = stack[0, DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
dark_values = dark_reference[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
finite = np.isfinite(image_values) & np.isfinite(dark_values)
fit_limit = np.percentile(image_values[finite], DARK_FIT_PERCENTILE)
used = finite & (image_values <= fit_limit)
scale, offset = dark_fits[0]

fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(dark_values[finite], image_values[finite], s=3, alpha=0.08,
             color="0.4", rasterized=True, label="sampled pixels")
axis.scatter(dark_values[used], image_values[used], s=4, alpha=0.25,
             color="tab:blue", rasterized=True, label="pixels used for fit")
x_line = np.linspace(dark_values[used].min(), dark_values[used].max(), 200)
axis.plot(x_line, scale * x_line + offset, color="red", linewidth=2,
          label=f"fit: y = {scale:.4g} x + {offset:.4g}")
axis.set_title(f"ID {IMAGE_ID}: first-frame dark fit")
axis.set_xlabel("Dark-reference intensity")
axis.set_ylabel("Raw-frame intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Inspect corrected intensities

Choose the thresholds in the next cell, then rerun that cell and the cleaning cell below.

In [13]:

plt.close("all")


# Edit these values while inspecting the histograms below.
INTENSITY_THRESHOLD = 20.0
HISTOGRAM_RANGE = (-20, 80)
HISTOGRAM_BINS = 400
PHOTON_VIEW_ROWS = slice(402, 900)
PHOTON_VIEW_COLUMNS = slice(420, 900)
gauss=False


representative_image = corrected_stack[0]
fig, axis = plt.subplots(figsize=(6, 4))
axis.hist(representative_image.ravel(), bins=HISTOGRAM_BINS, range=HISTOGRAM_RANGE, histtype="step")
axis.axvline(INTENSITY_THRESHOLD, color="red", linestyle="--", label="threshold")
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Intensity")
axis.set_ylabel("Pixel count")
axis.set_xlim(*HISTOGRAM_RANGE)
axis.legend()
axis.set_yscale("log")
plt.tight_layout()
plt.show()

import scipy
def gauss(image, sigma=3):
    return scipy.ndimage.gaussian_filter(image, sigma=sigma)
    
# Inspect the same first frames on the scale of individual photon events.
fig, axis = plt.subplots(figsize=(6, 5))
photon_view = corrected_stack[0, PHOTON_VIEW_ROWS, PHOTON_VIEW_COLUMNS]
if gauss:
    image = axis.imshow(
    photon_view*(gauss(photon_view, 2)>(INTENSITY_THRESHOLD/2)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
else:
    image = axis.imshow(
    photon_view*((photon_view)>(INTENSITY_THRESHOLD)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
    
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Column in selected region")
axis.set_ylabel("Row in selected region")
fig.colorbar(image, ax=axis, label="Corrected intensity")
plt.tight_layout()
plt.show()

## Threshold, clip, and average

In [7]:
plt.close("all")


In [8]:
# Subtract the chosen threshold from every frame, clip negatives, then average.
if gauss:
    blurred_stack=corrected_stack.copy()
    for i in range(blurred_stack.shape[0]):
        blurred_stack[i]=corrected_stack[i]*(gauss(corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
    cleaned_stack = np.clip(blurred_stack - 0*float(INTENSITY_THRESHOLD), 0, None)
else:
    blurred_stack=corrected_stack.copy()
    cleaned_stack = np.clip(blurred_stack - float(INTENSITY_THRESHOLD), 0, None)


cleaned_average = np.mean(cleaned_stack, axis=0, dtype=np.float64)
print("Threshold:", INTENSITY_THRESHOLD, "average shape:", cleaned_average.shape)

# Compare the blind average returned by load_processing with the cleaned average.
# Each column uses one shared linear color scale so the change is directly visible.
AVERAGE_DISPLAY_PERCENTILES = (1, 10.9)

comparison_values = np.concatenate((blind_average.ravel(), cleaned_average.ravel()))
comparison_values = comparison_values[np.isfinite(comparison_values)]
vmin, vmax = np.percentile(comparison_values, AVERAGE_DISPLAY_PERCENTILES)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))


for axis, image, description in zip(
    axes,
    (blind_average, cleaned_average),
    ("blind load_processing average", "cleaned average"),
):
    shown = axis.imshow(image, vmin=vmin, vmax=vmax, cmap="viridis")
    axis.set_title(f"ID {IMAGE_ID}: {description} (linear scale)")
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, label="Average intensity")
plt.tight_layout()
plt.show()

Threshold: 20.0 average shape: (2048, 2048)


In [9]:
BASEFOLDER="/home/experiences/sextants/com-sextants/SEXT_NEW"

## Save one cleaned average per acquisition

In [10]:
plt.close("all")

OUTPUT_FOLDER = BASEFOLDER + "/processed/" + "cleaned_acquisitions"
USER = "rb"

setup_metadata = loader.load(IMAGE_ID).metadata
output_file = OUTPUT_FOLDER + f"cleaned_ImId_{IMAGE_ID:04d}_{USER}.npz"
np.savez_compressed(
    output_file,
    image=cleaned_average,
    blind_average=blind_average,
    image_id=np.asarray(IMAGE_ID, dtype=int),
    dark_ids=np.asarray(DARK_IDS, dtype=int),
    threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
    dark_fits=dark_fits,
    energy_eV=float(setup_metadata["energy_eV"]),
    ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
    px_size_m=11.0e-6,
)
print(f"Saved ID {IMAGE_ID}: {output_file}")

Saved ID 495: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitionscleaned_ImId_0495_rb.npz


In [17]:
plt.close("all")

# After tuning the parameters above, put the remaining image IDs here.
# This repeats loading, dark subtraction, thresholding, averaging, and saving
# without producing diagnostic plots. An empty list does nothing.
BATCH_IMAGE_IDS = list(np.arange(525, 529) )

for batch_image_id in BATCH_IMAGE_IDS:
    batch_blind_average, batch_stack = load_processing(loader, batch_image_id)

    batch_corrected_frames = []
    batch_dark_fits = []
    for frame in batch_stack:
        if FIT_DARK_LINEAR:
            corrected, scale, offset = subtract_fitted_dark(
                frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
                DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
            )
        else:
            corrected, scale, offset = frame - dark_reference, 1.0, 0.0
        batch_corrected_frames.append(corrected)
        batch_dark_fits.append((scale, offset))

    batch_corrected_stack = np.stack(batch_corrected_frames)

    if gauss:
        blurred_stack=batch_corrected_stack.copy()
        for i in range(blurred_stack.shape[0]):
            blurred_stack[i]=batch_corrected_stack[i]*(gauss(batch_corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
        batch_cleaned_stack = np.clip(blurred_stack- 0*float(INTENSITY_THRESHOLD), 0, None)
    else:
        blurred_stack=batch_corrected_stack.copy()
        batch_cleaned_stack = np.clip(blurred_stack- float(INTENSITY_THRESHOLD), 0, None)

    batch_cleaned_average = np.mean(
        batch_cleaned_stack, axis=0, dtype=np.float64
    )
    batch_output_file = (
        OUTPUT_FOLDER + f"/cleaned_ImId_{batch_image_id:04d}_{USER}.npz"
    )
    np.savez_compressed(
        batch_output_file,
        image=batch_cleaned_average,
        blind_average=batch_blind_average,
        image_id=np.asarray(batch_image_id, dtype=int),
        dark_ids=np.asarray(DARK_IDS, dtype=int),
        threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
        dark_fits=np.asarray(batch_dark_fits),
        energy_eV=float(setup_metadata["energy_eV"]),
        ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
        px_size_m=11.0e-6,
    )
    print(f"Saved ID {batch_image_id}: {batch_output_file}")

print("fine-tuned im_id:", IMAGE_ID)
print("batch im_ids:", BATCH_IMAGE_IDS)
print("dark_ids:", DARK_IDS)

Saved ID 525: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0525_rb.npz
Saved ID 526: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0526_rb.npz
Saved ID 527: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0527_rb.npz
Saved ID 528: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0528_rb.npz
fine-tuned im_id: 495
batch im_ids: [np.int64(525), np.int64(526), np.int64(527), np.int64(528)]
dark_ids: [488]
